# Serial Communication and Robot Control for the myCobot 280 Labs
## Lab 7: Frames, Commands, State Feedback, and Safe Real-Robot Operation

Run this notebook from `~/venvs/mycobot`:

```bash
source ~/venvs/mycobot/bin/activate
jupyter lab
```

This is the first lab where you interact directly with the physical myCobot 280. You will inspect the serial protocol, send commands, read back robot state, compare commanded and measured motion, and use basic safety-stop procedures. The notebook connects low-level serial communication to the higher-level ROS 2 workflow.

**Important:** This notebook has two parts. In Part 1 (Tasks 1–15), you work with the serial interface directly — the ROS 2 controller must NOT be running. In Part 2 (Tasks 16–29), you use the ROS 2 controller — the serial object must be closed first.

**Ethernet setup for this lab:** the robot is reached with `ssh cobot@129.217.130.11`. Run the direct-serial section in a Python kernel on the robot computer; an IP address is not a serial device path. Select the serial device visible there in Task 2. The current SSH banner timeout must be resolved before collecting hardware results. No measured results are supplied or simulated in this notebook.

In [1]:
import importlib.util
from IPython import get_ipython
import os
import time
import numpy as np

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "widget" if importlib.util.find_spec("ipympl") else "inline")
import matplotlib.pyplot as plt

def joint_vector(values, label="Joint vector"):
    if values is None:
        raise RuntimeError(f"{label}: no measurement received; check the connection first.")
    q = np.asarray(values, dtype=float)
    if q.shape != (6,) or not np.all(np.isfinite(q)):
        raise ValueError(f"{label} must contain six finite joint values.")
    return q

The serial interface and ROS 2 imports are loaded in their respective sections below to keep the two parts cleanly separated.

## Part 1: Direct Serial Communication

In this part you communicate directly with the myCobot 280 through the serial interface. The ROS 2 controller (`mycobot_control`) must **not** be running — it would hold the serial port exclusively.

## Setup and Codebase Inspection

### 1. Inspect the codebase files

Before running any commands, read through the following files in this workspace:
- `serial_iface.py` — the low-level serial wrapper
- `mycobot_control.py` — the ROS 2 controller node (used in Part 2)

Look for:
- How the serial port is opened and configured
- How frames are assembled and parsed
- How angles are encoded (int16, degrees × 100)
- How stop/resume commands are structured
- How the ROS 2 node bridges serial commands to topics

**Inspection answer:** `MyCobotSerialInterface` opens pyserial with the configured device, baud rate and read/write timeouts. A lock serializes transactions. Frames contain a header, length, command, payload and tail; the read routine checks framing, command and response length. Joint angles use signed, big-endian 16-bit integers in hundredths of a degree. The wrapper sends stop/resume opcodes, while the ROS controller translates topic messages into serial requests and publishes joint state in radians.

In [2]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "notebooks"]
notebook_dir = next((p for p in candidates if (p / "serial_iface.py").is_file()), None)
if notebook_dir is None:
    raise FileNotFoundError("Open the workspace or notebooks directory containing serial_iface.py.")
sys.path.insert(0, str(notebook_dir.resolve()))
serial_iface_path = notebook_dir / "serial_iface.py"
print("Serial wrapper:", serial_iface_path.resolve())
print("Controller source: src/mycobot_control/mycobot_control/mycobot_control.py")

Serial wrapper: /home/corobot/mycobot_ws/src/mycobot_control/mycobot_control/serial_iface.py
Controller source: src/mycobot_control/mycobot_control/mycobot_control.py


### 2. Define the serial connection parameters

Select the robot's actual serial device on the machine running the Jupyter kernel. `/dev/ttyAMA0` is an onboard UART path, not a universal myCobot port. A USB connection may appear as `/dev/ttyUSB0`, `/dev/ttyACM0`, or a stable `/dev/serial/by-id/...` path.

Run the cell to list ports, then set `serial_port` to the robot's device and rerun it. Alternatively, set `MYCOBOT_PORT` before starting Jupyter. Ports are listed without opening them; the program does not assume that an arbitrary serial device is the robot. If the robot is connected to another computer, run the kernel there or make the device available on this host. Confirm the baud rate for your robot's connection; the lab UART default below is retained.

In [3]:
import os
from serial.tools import list_ports

available_ports = sorted(list_ports.comports(), key=lambda item: item.device)
print("Serial ports visible to this Jupyter kernel:")
for info in available_ports:
    print(f"  {info.device}: {info.description} [{info.hwid}]")
if not available_ports:
    print("  No serial ports detected. Connect the robot and rerun this cell.")

# Set this explicitly after identifying the robot in the list above.
# Example only: serial_port = "/dev/ttyUSB0"
serial_port = os.environ.get("MYCOBOT_PORT", "").strip()
baud_rate = 1_000_000  # Lab UART default; confirm for your connection.
timeout_s = 0.05
print(f"Selected port: {serial_port or '(not set)'}, baud: {baud_rate}")

Serial ports visible to this Jupyter kernel:
  /dev/ttyS0: n/a [n/a]
  /dev/ttyS1: n/a [n/a]
  /dev/ttyS10: n/a [n/a]
  /dev/ttyS11: n/a [n/a]
  /dev/ttyS12: n/a [n/a]
  /dev/ttyS13: n/a [n/a]
  /dev/ttyS14: n/a [n/a]
  /dev/ttyS15: n/a [n/a]
  /dev/ttyS16: n/a [n/a]
  /dev/ttyS17: n/a [n/a]
  /dev/ttyS18: n/a [n/a]
  /dev/ttyS19: n/a [n/a]
  /dev/ttyS2: n/a [n/a]
  /dev/ttyS20: n/a [n/a]
  /dev/ttyS21: n/a [n/a]
  /dev/ttyS22: n/a [n/a]
  /dev/ttyS23: n/a [n/a]
  /dev/ttyS24: n/a [n/a]
  /dev/ttyS25: n/a [n/a]
  /dev/ttyS26: n/a [n/a]
  /dev/ttyS27: n/a [n/a]
  /dev/ttyS28: n/a [n/a]
  /dev/ttyS29: n/a [n/a]
  /dev/ttyS3: n/a [n/a]
  /dev/ttyS30: n/a [n/a]
  /dev/ttyS31: n/a [n/a]
  /dev/ttyS4: n/a [n/a]
  /dev/ttyS5: n/a [n/a]
  /dev/ttyS6: n/a [n/a]
  /dev/ttyS7: n/a [n/a]
  /dev/ttyS8: n/a [n/a]
  /dev/ttyS9: n/a [n/a]
Selected port: (not set), baud: 1000000


## Communication Frames and Packet Structure

The myCobot 280 uses structured frames so the controller can identify command boundaries, interpret payloads, and detect invalid data. The frame format is:

```
FE FE LEN CMD [payload] FA
```

- `FE FE` — header (2 bytes)
- `LEN` — byte count from CMD through FA inclusive: `LEN = 1 + len(payload) + 1`
- `CMD` — command type byte
- `[payload]` — command-specific data (may be empty)
- `FA` — tail byte

### 3. Define the protocol constants

Define the header, tail, and key command bytes from the codebase.

In [4]:
HEAD = 0xFE
TAIL = 0xFA

CMD_READ_ANGLES = 0x20
CMD_SEND_ANGLES = 0x22
CMD_RESUME = 0x28
CMD_STOP = 0x29

print(f'Header: 0x{HEAD:02X}, Tail: 0x{TAIL:02X}')
print(f'Read angles: 0x{CMD_READ_ANGLES:02X}')
print(f'Send angles: 0x{CMD_SEND_ANGLES:02X}')
print(f'Stop: 0x{CMD_STOP:02X}, Resume: 0x{CMD_RESUME:02X}')

Header: 0xFE, Tail: 0xFA
Read angles: 0x20
Send angles: 0x22
Stop: 0x29, Resume: 0x28


### 4. Manually assemble a frame

Build the **stop** frame by hand. A stop command has no payload, so `LEN = 1 (CMD) + 1 (FA) = 2`. The full frame is: `FE FE 02 29 FA`.

Also build the **read-angles** request frame: `FE FE 02 20 FA`.

In [5]:
# Stop frame
stop_frame = bytes([HEAD, HEAD, 0x02, CMD_STOP, TAIL])  # Build the protocol frame for the stop command
print('Stop frame:', stop_frame.hex(' '))

# Read-angles request frame
read_frame = bytes([HEAD, HEAD, 0x02, CMD_READ_ANGLES, TAIL])  # Build the protocol frame for requesting joint angles
print('Read frame:', read_frame.hex(' '))

Stop frame: fe fe 02 29 fa
Read frame: fe fe 02 20 fa


### 5. Write a frame-printing helper

Create a short function that prints any byte sequence as formatted hex for debugging. This is useful throughout the lab when inspecting raw serial traffic.

In [6]:
def print_frame(label, frame_bytes):
    hex_str = ' '.join(f'{b:02X}' for b in frame_bytes)
    print(f'{label}: [{hex_str}] ({len(frame_bytes)} bytes)')

# Test with the stop frame
print_frame('Stop', stop_frame)
print_frame('Read', read_frame)

Stop: [FE FE 02 29 FA] (5 bytes)
Read: [FE FE 02 20 FA] (5 bytes)


## Reading Robot State

### 6. Open the serial connection

Instantiate the `MyCobotSerialInterface` from the codebase wrapper. This opens the serial port — make sure the ROS 2 controller is NOT running.

**Note:** The serial interface uses an internal threading lock. Do not run multiple serial cells simultaneously.

In [7]:
from pathlib import Path
import serial
from serial_iface import MyCobotSerialInterface

if not isinstance(serial_port, str) or not serial_port.strip():
    raise ValueError(
        "Set serial_port in Task 2 to the robot's device from the port list, "
        "then rerun Task 2 and this cell."
    )
if os.name == "posix" and not Path(serial_port).exists():
    raise FileNotFoundError(
        f"Serial device {serial_port!r} does not exist on the Jupyter kernel host. "
        "Connect the robot, rerun Task 2, and select its actual device. "
        "For a remote kernel, the device must be connected to that host."
    )

# Release this notebook's previous connection before opening it again.
if "ser" in globals():
    ser.close()
    del ser
try:
    ser = MyCobotSerialInterface(
        port=serial_port,
        baud=baud_rate,
        timeout=timeout_s,
        num_joints=6,
        stop_opcode=CMD_STOP,
        resume_opcode=CMD_RESUME,
    )
except serial.SerialException as exc:
    raise RuntimeError(
        f"Cannot open {serial_port!r}: {exc}. Check the device connection, "
        "serial access permissions, and that the ROS 2 controller has released it."
    ) from exc
print(f"Serial port opened: {serial_port}")

ValueError: Set serial_port in Task 2 to the robot's device from the port list, then rerun Task 2 and this cell.

### 7. Read the current joint angles

Call `read_angles_deg()` to read the measured joint state. The returned values are in **degrees** (the hardware encoding divides int16 values by 100). These are measured encoder values, not commanded targets.

In [ ]:
measured_deg = ser.read_angles_deg(timeout_s=0.5)  # Read the current joint angles from the robot

if measured_deg is not None:
    print('Measured joint angles (deg):', measured_deg)
else:
    print('No response from robot. Check connection and power.')

### 8. Convert and compare with a reference

Convert the measured angles to a NumPy array and compare with the expected home position (all zeros). Note: the myCobot 280 rarely reads exactly zero due to encoder resolution and mechanical play.

In [ ]:
if measured_deg is not None:
    measured_arr = np.array(measured_deg)
    home_deg = np.zeros(6)
    diff_deg = measured_arr - home_deg  # Compute the difference between the measured and home configurations

    print('Measured (deg):', measured_arr)
    print('Home (deg):    ', home_deg)
    print('Difference (deg):', diff_deg)

## Sending Position Commands

### 9. Define a safe position command

Define a small, safe joint-angle target in **degrees** and a speed level (1–100). Start with a very small motion from the current position.

**Caution:** Ensure the workspace is clear and the motion direction is understood before sending any command.

In [ ]:
# A small relative target still requires checking joint limits and clearance.
measured_deg = ser.read_angles_deg(timeout_s=0.5)
current_deg = joint_vector(measured_deg, "Current angles (deg)")
target_deg = current_deg + np.array([10.0, 0, 0, 0, 0, 0])
speed = 20
print("Target (deg):", target_deg)
print("Speed:", speed)

### 10. Send the position command

Send the target using the codebase wrapper. Start with small, well-understood motions and confirm the result before attempting larger movements.

In [ ]:
ser.send_angles_deg(target_deg, speed)
print('Position command sent.')

# Wait for the motion to complete
time.sleep(3.0)

### 11. Validate: compare commanded and measured angles

Read the robot state again after the motion and compute the difference between the commanded target and the measured position. Small residual differences are normal.

In [ ]:
measured_after = ser.read_angles_deg(timeout_s=0.5)

if measured_after is not None:
    measured_after_arr = np.array(measured_after)
    target_arr = np.array(target_deg)
    error_deg = measured_after_arr - target_arr  # Compute the command-to-measurement error

    print('Commanded (deg):', target_arr)
    print('Measured (deg): ', measured_after_arr)
    print('Error (deg):    ', error_deg)
    print('Max |error| (deg):', np.max(np.abs(error_deg)))
else:
    raise RuntimeError("No post-command measurement received; motion completion is unknown.")

### 12. Plot commanded vs measured joint values

Visualise the commanded and measured values side by side for all six joints.

In [ ]:
if measured_after is not None:
    joints = np.arange(1, 7)
    fig, ax = plt.subplots(figsize=(8, 4))
    width = 0.3
    ax.bar(joints - width/2, target_arr, width, label='Commanded')
    ax.bar(joints + width/2, measured_after_arr, width, label='Measured')
    ax.set_xlabel('Joint')
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Commanded vs Measured Joint Angles')
    ax.legend()
    ax.grid(True, axis='y')
    plt.tight_layout()

## Stop and Resume Handling

### 13. Send a stop command

The stop command immediately halts robot motion. Use the codebase wrapper method — this is not optional but an essential part of safe robot operation.

In [ ]:
ser.stop_motion()
print('Stop command sent.')

### 14. Send a resume command

After a stop, the robot ignores motion commands until a resume is issued. Send the resume command and confirm that the robot accepts commands again.

In [ ]:
ser.resume_motion()
print('Resume command sent.')

# Verify by reading angles
check = ser.read_angles_deg(timeout_s=0.5)
if check is not None:
    print('Readback received (does not prove motion resumed). Angles (deg):', check)
else:
    print('No response after resume.')

### 15. Close the serial connection

Before switching to the ROS 2 controller in Part 2, close the serial port. The controller needs exclusive access to the serial device.

**Reflection:** Why should stop functionality be treated as part of the normal control workflow, not as an exceptional feature? Consider what happens if you issue a command with an unintended target or if the robot moves in an unexpected direction.

**Answer:** stopping belongs in every motion workflow because incorrect targets, lost feedback or unexpected motion can occur during normal testing. A software stop depends on communication and controller response, so the physical stop mechanism must also remain accessible.

In [ ]:
if "ser" in globals():
    ser.close()
    del ser
print("Notebook serial connection closed. The ROS 2 controller can now use the port.")

## Part 2: ROS 2 Controller Integration

In this part you use the ROS 2 controller node (`mycobot_control`) which handles serial communication internally. Launch the controller in a separate terminal before proceeding:

```bash
source ~/venvs/mycobot/bin/activate
ros2 launch mycobot_control mycobot_control.launch.py
```

**Unit convention change:** The direct serial interface uses **degrees** (hardware encoding). The ROS 2 topics use **radians** (ROS convention). The controller converts internally:
- `/mycobot/joint_command`: expects **radians** → controller does `math.degrees(x)`
- `/mycobot/joint_velocity`: expects **rad/s** → mapped to speed 0–100 internally
- `/mycobot/joint_states`: publishes positions in **radians**
- `/mycobot/ee_pose`: expects `[x_mm, y_mm, z_mm, roll_deg, pitch_deg, yaw_deg]`

## Position Mode and Velocity Mode

### 16. Position mode vs velocity mode

**Position mode** (`/mycobot/joint_command`): you specify a target configuration, and the controller drives the robot there. Appropriate when the desired end state is known.

**Velocity mode** (`/mycobot/joint_velocity`): you specify a motion rate, and the robot moves as long as the velocity command is active. Appropriate for continuous or incremental motion. Requires extra care:
- Deadband: velocities below 0.01 rad/s per joint are treated as zero
- Max speed: ~2.618 rad/s (150 deg/s)
- Zero velocity vector triggers an immediate stop
- Acceleration ramp: limited to 1.0 rad/s²

### 17. Inspect the ROS 2 topics

Use the ROS 2 CLI to verify the controller is running and inspect the available topics.

In [ ]:
# Run these in a terminal or uncomment to run from the notebook:
# !ros2 topic list
# !ros2 topic info /mycobot/joint_states
# !ros2 topic info /mycobot/joint_command
# !ros2 topic info /mycobot/joint_velocity
# !ros2 topic echo /mycobot/joint_states --once

## ROS 2 Node Integration

### 18. Create a notebook node

Create a small ROS 2 node that subscribes to `/mycobot/joint_states` and stores the most recent measured state.

In [ ]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float64MultiArray


class SerialLabNode(Node):
    def __init__(self):
        super().__init__('serial_lab_node')
        self.last_js = None
        self.js_sub = self.create_subscription(
            JointState, '/mycobot/joint_states', self._js_cb, 10,
        )

    def _js_cb(self, msg):
        self.last_js = msg

def ordered_positions(msg):
    if msg is None:
        raise RuntimeError("No joint state received. Check the ROS 2 controller.")
    if len(msg.name) != len(msg.position) or len(set(msg.name)) != len(msg.name):
        raise ValueError("JointState names and positions are inconsistent.")
    positions = dict(zip(msg.name, msg.position))
    names = [f"joint{i}" for i in range(1, 7)]
    if any(name not in positions for name in names):
        raise ValueError("JointState must contain joint1 through joint6.")
    return joint_vector([positions[name] for name in names], "JointState positions")

def wait_for_joint_state(node, timeout=2.0):
    # Wait for a new callback rather than reusing the previously stored state.
    node.last_js = None
    deadline = time.monotonic() + timeout
    while node.last_js is None and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=0.1)
    return ordered_positions(node.last_js)

### 19. Initialize and spin

Initialize `rclpy`, create the node, and spin once to receive the latest joint state.

In [ ]:
if "node" in globals() and rclpy.ok():
    node.destroy_node()
if not rclpy.ok():
    rclpy.init(args=None)
node = SerialLabNode()
q_observed = wait_for_joint_state(node)
print("Positions joint1–joint6 (rad):", q_observed)
print("Positions joint1–joint6 (deg):", np.rad2deg(q_observed))
# Recreate publishers after rerunning this cell.

### 20. Create a position-command publisher

Create a publisher for `/mycobot/joint_command`. This topic expects joint angles in **radians**.

In [ ]:
pos_pub = node.create_publisher(Float64MultiArray, '/mycobot/joint_command', 10)

### 21. Publish a safe position command (radians)

Send a small safe joint-position command in **radians** via the ROS 2 position-mode path. This is the higher-level equivalent of the direct `send_angles_deg()` used in Part 1.

In [ ]:
q_cmd_rad = wait_for_joint_state(node) + np.array([np.deg2rad(5), 0, 0, 0, 0, 0])
msg = Float64MultiArray()
msg.data = joint_vector(q_cmd_rad).tolist()
pos_pub.publish(msg)
print("Published position command (rad):", q_cmd_rad)
# Process callbacks while waiting so the subscription queue is drained.
deadline = time.monotonic() + 3.0
while time.monotonic() < deadline:
    rclpy.spin_once(node, timeout_sec=0.1)
q_after = wait_for_joint_state(node)
print("Measured (rad):", q_after)
print("Measured minus commanded (deg):", np.rad2deg(q_after - q_cmd_rad))

### 22. Create a velocity-command publisher

Create a publisher for `/mycobot/joint_velocity`. This topic expects joint velocities in **rad/s**.

In [ ]:
vel_pub = node.create_publisher(Float64MultiArray, '/mycobot/joint_velocity', 10)

### 23. Publish a safe velocity command

Send a very small velocity on one joint for a short duration, then send zeros to stop.

**Safety warning:**
- Keep velocities small (e.g., 0.1 rad/s on one joint)
- Keep duration short (e.g., 1–2 seconds)
- Be ready to send zeros or use the stop mechanism immediately
- Velocities below 0.01 rad/s are ignored (deadband)
- Sending `[0, 0, 0, 0, 0, 0]` triggers an immediate stop

In [ ]:
qdot_cmd = joint_vector([0.1, 0, 0, 0, 0, 0], "Velocity (rad/s)")
duration = 1.0
rate_hz = 20.0
msg = Float64MultiArray()
wait_for_joint_state(node)  # Require feedback before starting.
start = time.monotonic()
try:
    while time.monotonic() - start < duration:
        msg.data = qdot_cmd.tolist()
        vel_pub.publish(msg)
        rclpy.spin_once(node, timeout_sec=0.0)
        time.sleep(1.0 / rate_hz)
finally:
    # Also attempt to stop if the loop raises or the cell is interrupted.
    msg.data = [0.0] * 6
    vel_pub.publish(msg)
    print("Zero-velocity stop command published.")

## Commanded vs Measured Motion

### 24. Record measured joint states over time

Send a position command and record the measured `/mycobot/joint_states` at regular intervals to observe the motion dynamics.

In [ ]:
q_target_rad = wait_for_joint_state(node) + np.array([np.deg2rad(5), 0, 0, 0, 0, 0])
msg = Float64MultiArray()
msg.data = joint_vector(q_target_rad).tolist()
pos_pub.publish(msg)

timestamps = []
measured_positions = []
node.last_js = None
start = time.monotonic()
record_duration = 4.0
while time.monotonic() - start < record_duration:
    rclpy.spin_once(node, timeout_sec=0.05)
    if node.last_js is not None:
        timestamps.append(time.monotonic() - start)
        measured_positions.append(ordered_positions(node.last_js))
        node.last_js = None  # Do not count one message repeatedly.
timestamps = np.asarray(timestamps)
measured_positions = np.asarray(measured_positions).reshape(-1, 6)
if len(timestamps) == 0:
    raise RuntimeError("No feedback recorded after command; check the controller.")
print(f"Recorded {len(timestamps)} callbacks in {record_duration} s")

### 25. Plot the motion over time

Visualise how each joint moves toward the commanded target over time.

In [ ]:
if len(timestamps) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    for j in range(measured_positions.shape[1]):
        ax.plot(timestamps, np.degrees(measured_positions[:, j]), label=f'q{j+1}')
        ax.axhline(np.degrees(q_target_rad[j]), color=f'C{j}', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Measured joint motion toward commanded target')
    ax.legend(fontsize=7)
    ax.grid(True)
    plt.tight_layout()

### 26. Interpret commanded vs measured differences

Why might commanded and measured values differ? Consider:
- Motion still in progress (robot has not yet reached the target)
- Sampling delay (discrete read intervals vs continuous motion)
- Internal controller limits (acceleration ramp, speed cap)
- Mechanical play and encoder resolution

In [ ]:
print("""Commanded angles are targets; feedback describes the controller's reported state.
During motion, speed and acceleration limits cause a time-dependent tracking error.
Serial polling, ROS delivery and notebook scheduling introduce sampling delays.
Encoder resolution and mechanical play can leave a residual error after settling.
The supplied controller can also publish estimated states, so verify the feedback
source before treating the plot as independent encoder measurement.
""")
if "measured_positions" in globals() and len(measured_positions):
    final_error_deg = np.rad2deg(measured_positions[-1] - q_target_rad)
    print("Last recorded error per joint (deg):", final_error_deg)
    print("Largest absolute final error (deg):", np.max(np.abs(final_error_deg)))
else:
    print("Hardware interpretation pending: run Task 24 after connectivity is restored.")

## Safety and Practical Precautions

### 27. Pre-motion safety checklist

Before enabling any robot motion, verify:
- Workspace is clear of obstacles and people
- Motion direction and magnitude are understood
- Stop mechanism is known and accessible
- Commands are modest during initial tests
- Serial port or controller is confirmed active

In [ ]:
checks = {
    "Workspace clear": False,
    "Target, joint limits and motion direction checked": False,
    "Physical stop accessible and software stop understood": False,
    "Speed and duration checked": False,
    "Serial connection or controller feedback verified": False,
}
for label, confirmed in checks.items():
    print(f"[{'x' if confirmed else ' '}] {label}")
print("Complete these checks before the motion cells in either part; this is a manual checklist.")

### 28. Why real-robot operation requires extra caution

Briefly explain why direct communication with a real robot requires more caution than simulated motion. Consider physical consequences, irreversible actions, and the gap between model assumptions and hardware behaviour.

In [ ]:
print("""A real robot can strike people, damage equipment or collide with itself.
Small joint changes are not automatically safe because tool reach and nearby
obstacles matter. Models omit some calibration errors, backlash and communication
delays. Check the actual workspace, joint limits and feedback; begin with modest
commands and keep a physical stop available. Software stop delivery is not guaranteed
if the connection fails. Simulation results alone do not establish hardware safety.""")

## Reflection

### 29. Summary reflection

Summarise briefly:
1. What does the serial protocol contribute to reliable robot interaction?
2. Why does the frame structure (`FE FE LEN CMD ... FA`) matter?
3. Why is comparing commanded vs measured motion essential?
4. When is position mode appropriate vs velocity mode?

In [ ]:
print("""1. The serial protocol defines how requests and angle data are encoded and
interpreted. Framing and validation help reject malformed replies, but this frame
format has no checksum and does not guarantee delivery or successful motion.
2. FE FE identifies a frame, LEN determines its extent, CMD selects the operation,
and FA terminates it. Signed angle encoding and units must agree at both ends.
3. Comparing targets and feedback reveals tracking delay, residual error and failed
commands. Confirm whether feedback is measured or estimated before interpreting it.
4. Position mode suits a known target configuration. Velocity mode suits continuous
rate control and needs a bounded duration and explicit stop. Direct serial angles
are degrees; ROS joint positions are radians and joint velocities are rad/s.""")

# Run cleanup when finished with Part 2:
# node.destroy_node()
# rclpy.shutdown()